# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font>  — Avance 6</center>

<center>

[![Materia](https://img.shields.io/badge/MATERIA-PROYECTO_INTEGRADOR-E0A800?style=for-the-badge&logoColor=white)](https://tec.mx)

</center>

<center>

[![Python](https://img.shields.io/badge/Python-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![GitHub](https://img.shields.io/badge/Repo-GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/jmtoral/proyecto_integrador_52)

</center>

## **<font color="#0036a3">Avance 6 — Análisis Cualitativo: Efecto del Image Enhancement bajo Sub/Sobre-exposición</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10</font>**

---

## **<center> <font color="#0036a3">Equipo 52</font> </center>**

<table style="border-collapse:collapse; width:60%; margin:auto;">
  <tr>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/elda.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>Elda Morales</strong><br><small>A00449074</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/mpgc.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>María Paula Gutiérrez</strong><br><small>A01747706</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/jmtc_n.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>José Manuel Toral</strong><br><small>A01122243</small>
    </td>
  </tr>
</table>

---

### Motivación

Las métricas globales (AbsRel, RMSE) promedian el error sobre los 550 fotogramas y **ocultan dónde** ayuda o estorba cada *enhancement*. En endoscopía, los dos regímenes de iluminación más problemáticos son:

- **Subexposición (*under*)** — zonas demasiado **oscuras** (cavidades profundas, regiones alejadas de la fuente de luz). La red pierde textura y tiende a colapsar la profundidad.
- **Sobreexposición (*over*)** — zonas **quemadas** por exceso de luz o reflejos especulares sobre tejido húmedo. La señal se satura y la geometría local se distorsiona.

Los métodos de *image enhancement* (EndoLMSPEC, IAT) buscan precisamente **corregir estos extremos**. Este avance los inspecciona de forma **cualitativa** sobre el mejor modelo, en un fotograma representativo de cada régimen.

### Modelo seleccionado: **MonoIIT**

MonoIIT es el modelo con **menor error** del estudio (AbsRel 0.0554 sobre el split oficial), consistente con la referencia del Dr. Espinosa. Usa encoder **MPViT** + decoder **HR-Depth** (con módulos de atención), cargado tal cual el repositorio oficial de MonoViT.

### Qué se muestra

Se inspeccionan **varios fotogramas representativos** del split, agrupados en cuatro regímenes (seleccionados automáticamente):
- **Subexpuestos** (los más oscuros) y **sobreexpuestos** (los más brillantes) — los extremos de iluminación.
- **Bien expuestos** (brillo cercano a la mediana) — referencia del comportamiento normal.
- **Con reflejos especulares marcados** (mayor área de píxeles saturados) — donde `endosttn` podría ayudar más.

Para cada fotograma se compara, en columnas: el **ground truth** (profundidad real del sensor SCARED), la imagen **original** (`none`) y cada **enhancement** (`endolmspec`, `iat`, `endosttn`), con el **mapa de profundidad** que MonoIIT predice de cada versión.

Los mapas se muestran en **disparidad** (1/profundidad), la convención de los trabajos de *depth*: **lo cercano sale claro y lo lejano oscuro**. Así se observa si la corrección de iluminación **acerca la predicción al ground truth**, o si introduce artefactos. (Los títulos de las figuras están en inglés.)

## Pipeline del Avance 6

Flujo completo: desde el fotograma crudo de SCARED, pasando por **cada algoritmo de realce**, la estimación de profundidad con **MonoIIT**, hasta la comparación contra el *ground truth* y las vistas cualitativas por régimen de iluminación.

```mermaid
flowchart TD
    A["SCARED<br/>rgb.mp4 + scene_points.tiff"] --> B["Extraccion del split oficial<br/>AF-SfMLearner — 550 frames, ds 1-7"]
    B --> C["INPUT<br/>fotograma corrupto (RGB 1280x1024)"]

    C --> ENH

    subgraph ENH ["Image Enhancement (4 metodos activos)"]
        direction TB
        E0["none<br/>(baseline, sin realce)"]
        E1["Endo-LMSPEC<br/>(deep · piramide Laplaciana + U-Nets)"]
        E2["IAT<br/>(deep · Illumination-Adaptive Transformer)"]
        E3["Endo-STTN<br/>(deep · temporal · inpainting de especulares)"]
        E4["retinex<br/>(clasico — DESACTIVADO)"]
    end

    E0 --> O["OUTPUT<br/>fotograma realzado"]
    E1 --> O
    E2 --> O
    E3 --> O
    E4 -. excluido .-> O

    O --> M["MonoIIT<br/>encoder MPViT + decoder HR-Depth + lighting"]
    M --> D["Mapa de profundidad predicho<br/>(median scaling a mm)"]

    GT["Ground truth SCARED<br/>(sensor estereo, disperso)"] --> CMP
    D --> CMP["Comparacion vs GT"]

    CMP --> R1["Vistas cualitativas por regimen<br/>under · over · bien expuesto · especular"]
    CMP --> R2["Metricas del estudio<br/>AbsRel · RMSE · delta"]

    style ENH fill:#cfe8ff,stroke:#2766CB,stroke-width:2px
    style M fill:#cfe8ff,stroke:#2766CB,stroke-width:2px
    style GT fill:#fff3cd,stroke:#E0A800,stroke-width:2px
    style R1 fill:#1a2e51,stroke:#E0A800,stroke-width:2px,color:#fff
    style R2 fill:#1a2e51,stroke:#E0A800,stroke-width:2px,color:#fff
    style E4 stroke-dasharray: 5 5,color:#999,stroke:#bbb
```

**Lectura del diagrama.** Cada fotograma del *split* oficial entra como imagen cruda y pasa por uno de los **cuatro métodos de realce activos** —`none` (baseline), **Endo-LMSPEC** e **IAT** (realce de una sola imagen) y **Endo-STTN** (realce temporal que reconstruye los reflejos especulares con frames vecinos). `retinex` se incluyó originalmente pero está **desactivado** por indicación del profesor (se conserva en el código). El fotograma realzado alimenta a **MonoIIT** (encoder MPViT + decoder HR-Depth con módulo de iluminación), que predice el mapa de profundidad; tras *median scaling* a milímetros se compara contra el **ground truth** del sensor estéreo de SCARED. De esa comparación salen las **vistas cualitativas por régimen** (subexpuesto, sobreexpuesto, bien expuesto y con especulares) y las **métricas** del estudio (AbsRel, RMSE, δ). Adaptado del pipeline de realce + reconstrucción para endoscopía, ajustado a los métodos y al modelo de **este** estudio.

In [3]:
from pathlib import Path
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab; IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE         = Path("/content/drive/MyDrive/proyecto_integrador")
    SCARED_ROOT  = BASE / "scared_raw"
    EDAM_PATH    = BASE / "Endo-Depth-and-Motion"
    LMSPEC_PATH  = BASE / "EndoLMSPEC"
    IAT_PATH     = BASE / "EndoViT"
    MONOVIT_PATH = BASE / "MonoViT"
    STTN_PATH    = BASE / "Endo-STTN"
    W            = BASE / "scared weights"
    REPO_ROOT    = Path("/content/repo_52")
    if not REPO_ROOT.exists():
        subprocess.check_call(["git","clone","--depth=1",
            "https://github.com/jmtoral/proyecto_integrador_52.git", str(REPO_ROOT)])
    else:
        subprocess.check_call(["git","-C",str(REPO_ROOT),"fetch","origin"])
        subprocess.check_call(["git","-C",str(REPO_ROOT),"reset","--hard","origin/main"])
    SPLIT_FILE = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
else:
    BASE         = Path("E:/scared_wights_complete/scared weights")
    SCARED_ROOT  = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH    = Path("E:/Endo-Depth-and-Motion")
    LMSPEC_PATH  = Path("E:/EndoLMSPEC")
    IAT_PATH     = Path("E:/EndoVit")
    MONOVIT_PATH = Path("E:/MonoViT")
    STTN_PATH    = Path("E:/Endo-STTN")
    W            = BASE
    REPO_ROOT    = Path(r"d:\Proyecto_Integrador\Corrreccion_Luz")
    SPLIT_FILE   = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"

W_MONOIIT      = W / "monoIIT_weights" / "trained-winner-weights"
LMSPEC_WEIGHTS = LMSPEC_PATH / "checkpoint" / "main_net" / "model_256_combined_SSIM5_1.pth"
IAT_WEIGHTS    = IAT_PATH / "Endo4IE" / "best_Epoch50_laplacian_histogan_loss.pth"
STTN_WEIGHTS   = STTN_PATH / "release_model" / "pretrained_model" / "gen_00009.pth"
NPZ_CACHE      = BASE / "split_frames.npz"

def load_split(sf):
    items=[]
    with open(sf) as f:
        for line in f:
            line=line.strip()
            if not line: continue
            folder, fid, _ = line.split()
            ds, kf = folder.split("/")
            items.append(("dataset_"+ds.replace("dataset",""), "keyframe_"+kf.replace("keyframe",""), int(fid)))
    return items
SPLIT_ITEMS = load_split(SPLIT_FILE)
from collections import defaultdict
SPLIT_BY_KF = defaultdict(list)
for ds,kf,fid in SPLIT_ITEMS: SPLIT_BY_KF[(ds,kf)].append(fid)
print(f"{'Colab' if IN_COLAB else 'Local'} | split {len(SPLIT_ITEMS)} frames")

Mounted at /content/drive
Colab | split 551 frames


In [4]:
import numpy as np
# Cargar imagenes + GT del npz cacheado
def _key(ds,kf,fid): return f"{ds}|{kf}|{fid}"
SPLIT_DATA = {}
_npz = np.load(NPZ_CACHE, allow_pickle=True)
for ds,kf,fid in SPLIT_ITEMS:
    k=_key(ds,kf,fid); ik,gk="img_"+k,"gt_"+k
    if ik in _npz.files:
        gt=_npz[gk] if gk in _npz.files else None
        if gt is not None and gt.size==1 and np.isnan(gt).all(): gt=None
        SPLIT_DATA[k]=(_npz[ik], gt)
def load_split_frame(ds,kf,fid):
    return SPLIT_DATA.get(_key(ds,kf,fid),(None,None))
print(f"Frames en memoria: {len(SPLIT_DATA)}")

Frames en memoria: 551


In [5]:
import torch, importlib.util as _ilu, types
import torch.nn as nn
import numpy as np
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_NETDIR = MONOVIT_PATH / "networks"
for _k in list(sys.modules):
    if _k=="networks" or _k.startswith("networks."): del sys.modules[_k]
_pkg=types.ModuleType("networks"); _pkg.__path__=[str(_NETDIR)]; sys.modules["networks"]=_pkg
def _ls(name,fn):
    sp=_ilu.spec_from_file_location(f"networks.{name}",str(_NETDIR/fn))
    m=_ilu.module_from_spec(sp); sys.modules[f"networks.{name}"]=m; sp.loader.exec_module(m); setattr(_pkg,name,m); return m
_ls("hr_layers","hr_layers.py")
_hr = _ls("hr_decoder","hr_decoder.py")
mpvit_small=_ls("mpvit","mpvit.py").mpvit_small
DepthDecoderHR = _hr.DepthDecoder

# MonoIIT usa encoder mpvit + decoder HR-Depth OFICIAL (convs.f4, X_00, attention),
# igual que MonoViT. Se carga con DepthDecoderHR() defaults (como evaluate_depth.py oficial).
enc = mpvit_small(); enc.num_ch_enc=[64,128,216,288,288]
_ed = torch.load(W_MONOIIT/"encoder.pth", map_location=DEVICE)
MH, MW = _ed.get("height",192), _ed.get("width",640)
enc.load_state_dict({k:v for k,v in _ed.items() if k in enc.state_dict()}); enc.to(DEVICE).eval()

_sd = torch.load(W_MONOIIT/"depth.pth", map_location=DEVICE)
dec = DepthDecoderHR()
_r = dec.load_state_dict(_sd, strict=False); dec.to(DEVICE).eval()
print(f"MonoIIT cargado {MH}x{MW} (HR-Depth) | missing={len(_r.missing_keys)} unexpected={len(_r.unexpected_keys)}")

import cv2, PIL.Image as pil
from torchvision import transforms
def predict_depth(img, max_depth=150.0, min_depth=0.1):
    H,W_=img.shape[:2]
    t=transforms.ToTensor()(pil.fromarray(img).resize((MW,MH),pil.LANCZOS)).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): out=dec(enc(t))
    disp=out[("disp",0)].squeeze().detach().cpu().numpy()
    sd=(1.0/max_depth)+((1.0/min_depth)-(1.0/max_depth))*disp
    sd=cv2.resize(sd,(W_,H)); return 1.0/sd

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


MonoIIT cargado 256x320 (HR-Depth) | missing=0 unexpected=0


In [6]:
# Cargar los enhancements (mismo codigo que el notebook principal)
import torchvision.transforms as T
subprocess.check_call([sys.executable,"-m","pip","install","-q","IQA_pytorch","path"])

def _load_endolmspec(p, device):
    _orig=sys.path.copy()
    clean=[str(p)]+[x for x in sys.path if "EndoSLAM" not in x and "endosfm" not in x.lower()
                    and "HADepth" not in x and "EndoViT" not in x and "EndoVit" not in x]
    for k in list(sys.modules):
        if k in ("utils","generator","unet") or k.startswith("utils."): del sys.modules[k]
    try:
        sys.path=clean
        sp=_ilu.spec_from_file_location("generator", p/"generator.py")
        mod=_ilu.module_from_spec(sp); sp.loader.exec_module(mod); G=mod.Generator
    finally: sys.path=_orig
    return G(n_channels=3, device=device, bilinear=False)
lmspec_net=_load_endolmspec(LMSPEC_PATH, DEVICE)
lmspec_net.load_state_dict(torch.load(LMSPEC_WEIGHTS, map_location=DEVICE)); lmspec_net.to(DEVICE).eval()

for k in list(sys.modules):
    if k=="utils" or k.startswith("utils."): del sys.modules[k]
sys.modules["imp"]=types.ModuleType("imp")
_sp=_ilu.spec_from_file_location("IAT_main_a5", IAT_PATH/"experiments"/"model"/"IAT_main.py")
_im=_ilu.module_from_spec(_sp)
if str(IAT_PATH/"experiments") not in sys.path: sys.path.insert(0,str(IAT_PATH/"experiments"))
_sp.loader.exec_module(_im)
iat_net=_im.IAT(in_dim=3, with_global=True, type="exp")
iat_net.load_state_dict(torch.load(IAT_WEIGHTS, map_location=DEVICE)); iat_net.to(DEVICE).eval()

def correct_none(img): return img
def correct_retinex(img, sigma=30):
    f=img.astype(np.float32)+1.0; r=np.zeros_like(f)
    for c in range(3):
        b=cv2.GaussianBlur(f[:,:,c],(0,0),sigma); r[:,:,c]=np.log(f[:,:,c])-np.log(b+1.0)
    r-=r.min(); return (r/(r.max()+1e-8)*255).astype(np.uint8)
def correct_endolmspec(img):
    t=T.ToTensor()(img).to(DEVICE)
    with torch.no_grad(): _,o=lmspec_net(t)
    return (o["subnet_16"][0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)
def correct_iat(img):
    t=torch.from_numpy(img.astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): _,_,e=iat_net(t)
    return (e[0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

CORRECTIONS={"none":correct_none,  # "retinex":correct_retinex,  # desactivado a peticion del profesor (conservado por si acaso)
             "endolmspec":correct_endolmspec,"iat":correct_iat}
print("Enhancements:", list(CORRECTIONS.keys()))
print("(endosttn se omite en esta visualizacion por ser temporal/pesado; se puede anadir si se desea)")

/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


Enhancements: ['none', 'endolmspec', 'iat']
(endosttn se omite en esta visualizacion por ser temporal/pesado; se puede anadir si se desea)


In [7]:
# --- Endo-STTN: inpainting temporal de especularidades (mismo codigo que Avance 5) ---
STTN_CKPT_DIR    = STTN_PATH / "release_model" / "pretrained_model"
STTN_GDRIVE_ID   = "14sdaDejsxgRuzHBSuqH2xEpbxuqyWI-R"
if IN_COLAB and not STTN_WEIGHTS.exists():
    STTN_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([sys.executable,"-m","pip","install","-q","gdown"])
    import gdown; gdown.download(id=STTN_GDRIVE_ID, output=str(STTN_WEIGHTS), quiet=False)

def _load_endo_sttn(sttn_path, ckpt, device):
    _orig_path = sys.path.copy()
    _orig_mods = {k: sys.modules[k] for k in list(sys.modules)
                  if k=="core" or k.startswith("core.") or k=="model" or k.startswith("model.")}
    for k in list(sys.modules):
        if k=="core" or k.startswith("core.") or k=="model" or k.startswith("model."): del sys.modules[k]
    try:
        sys.path.insert(0, str(sttn_path))
        net_mod = __import__("model.sttn", fromlist=["InpaintGenerator"])
        utils_mod = __import__("core.utils", fromlist=["Stack","ToTorchFormatTensor"])
        model = net_mod.InpaintGenerator().to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device)["netG"]); model.eval()
        Stack = utils_mod.Stack; ToTorch = utils_mod.ToTorchFormatTensor
    finally:
        sys.path = _orig_path
        for k in list(sys.modules):
            if k=="core" or k.startswith("core.") or k=="model" or k.startswith("model."): del sys.modules[k]
        sys.modules.update(_orig_mods)
    return model, Stack, ToTorch

STTN_W, STTN_H = 288, 288
STTN_REF_LEN, STTN_STRIDE = 10, 5
_sttn_ok = STTN_WEIGHTS.exists()
if _sttn_ok:
    sttn_model, _Stack, _ToTorch = _load_endo_sttn(STTN_PATH, STTN_WEIGHTS, DEVICE)
    _sttn_to_tensors = T.Compose([_Stack(), _ToTorch()])
    print("Endo-STTN OK")
else:
    print("Endo-STTN: pesos no encontrados; se omitira esa columna")

def _sttn_specular_mask(img_rgb, dil=8):
    L = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)[:,:,0].astype(np.float32)
    m = (L >= np.percentile(L, 97)).astype(np.uint8)
    if dil: m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(dil,dil)), iterations=1)
    return m

def _sttn_get_ref_index(nb_ids, length):
    return [i for i in range(0, length, STTN_REF_LEN) if i not in nb_ids]

@torch.no_grad()
def endo_sttn_inpaint_sequence(frames_rgb):
    if not _sttn_ok: return frames_rgb
    H0, W0 = frames_rgb[0].shape[:2]
    pil_frames = [pil.fromarray(f).resize((STTN_W,STTN_H), pil.LANCZOS) for f in frames_rgb]
    masks_np   = [cv2.resize(_sttn_specular_mask(f),(STTN_W,STTN_H),interpolation=cv2.INTER_NEAREST) for f in frames_rgb]
    pil_masks  = [pil.fromarray((m*255).astype(np.uint8)) for m in masks_np]
    vlen = len(pil_frames)
    feats = _sttn_to_tensors(pil_frames).unsqueeze(0)*2-1
    masks = _sttn_to_tensors(pil_masks).unsqueeze(0)
    feats, masks = feats.to(DEVICE), masks.to(DEVICE)
    bin_masks = [np.expand_dims((np.array(m)!=0).astype(np.uint8),2) for m in pil_masks]
    raw_frames = [np.array(f).astype(np.uint8) for f in pil_frames]
    feats = sttn_model.encoder((feats*(1-masks).float()).view(vlen,3,STTN_H,STTN_W))
    _, c, fh, fw = feats.size(); feats = feats.view(1, vlen, c, fh, fw)
    comp = [None]*vlen
    for f in range(0, vlen, STTN_STRIDE):
        nb_ids = [i for i in range(max(0,f-STTN_STRIDE), min(vlen,f+STTN_STRIDE+1))]
        ref_ids = _sttn_get_ref_index(nb_ids, vlen)
        pred_feat = sttn_model.infer(feats[0, nb_ids+ref_ids], masks[0, nb_ids+ref_ids])
        pred_img = torch.tanh(sttn_model.decoder(pred_feat[:len(nb_ids)]))
        pred_img = ((pred_img+1)/2).cpu().permute(0,2,3,1).numpy()*255
        for i, idx in enumerate(nb_ids):
            im = pred_img[i].astype(np.uint8)*bin_masks[idx] + raw_frames[idx]*(1-bin_masks[idx])
            comp[idx] = im if comp[idx] is None else (comp[idx]*0.5 + im*0.5).astype(np.uint8)
    return [cv2.resize(c, (W0, H0), interpolation=cv2.INTER_LANCZOS4) for c in comp]

_STTN_CACHE = {}
def correct_endo_sttn(img_rgb, ds=None, kf=None, fid=None):
    if not _sttn_ok: return img_rgb
    if ds is None: return endo_sttn_inpaint_sequence([img_rgb])[0]
    key = (ds, kf)
    if key not in _STTN_CACHE:
        fids = SPLIT_BY_KF[key]
        seq = [load_split_frame(ds, kf, f)[0] for f in fids]
        out = endo_sttn_inpaint_sequence(seq)
        _STTN_CACHE.clear(); _STTN_CACHE[key] = {f:o for f,o in zip(fids, out)}
    return _STTN_CACHE[key][fid]

if _sttn_ok:
    CORRECTIONS["endosttn"] = correct_endo_sttn
print("Enhancements:", list(CORRECTIONS.keys()))

Endo-STTN OK
Enhancements: ['none', 'endolmspec', 'iat', 'endosttn']


---
## Selección automática de fotogramas *under* y *over*

Para cada fotograma del split se calcula su **brillo medio** (canal L de CIELAB). El fotograma con menor brillo representa el caso **subexpuesto** y el de mayor brillo el **sobreexpuesto**. Sobre cada uno se aplican los *enhancements* y se obtiene el mapa de profundidad de MonoIIT.

In [ ]:
import matplotlib.pyplot as plt
import numpy.ma as ma
%matplotlib inline

# --- Metricas por frame: brillo medio (L de LAB) y % de pixeles especulares ---
def _brillo(img):
    return cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[:,:,0].mean()

def _pct_especular(img, thr=240):
    L = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[:,:,0]
    return (L >= thr).mean()*100.0

# Caracterizar todos los frames del split una sola vez
_stats = []
for ds,kf,fid in SPLIT_ITEMS:
    img,gt = load_split_frame(ds,kf,fid)
    if img is not None and gt is not None:
        _stats.append({"b":_brillo(img), "spec":_pct_especular(img), "ds":ds, "kf":kf, "fid":fid})
_stats.sort(key=lambda s: s["b"])

_b_med = np.median([s["b"] for s in _stats])

# --- Seleccion de casos (sin duplicados), con etiqueta descriptiva en ingles ---
N_UNDER, N_OVER, N_MED, N_SPEC = 3, 3, 2, 2
_seen = set()
CASOS = []   # (label_en, regimen_slug, stat)
def _caso_key(s): return (s["ds"], s["kf"], s["fid"])
def _add(label, slug, s):
    if _caso_key(s) in _seen: return False
    _seen.add(_caso_key(s)); CASOS.append((label, slug, s)); return True

# Top-N mas oscuros (under) y mas brillantes (over)
for s in _stats:
    if len([c for c in CASOS if c[1]=="under"]) >= N_UNDER: break
    _add("Underexposed", "under", s)
for s in reversed(_stats):
    if len([c for c in CASOS if c[1]=="over"]) >= N_OVER: break
    _add("Overexposed", "over", s)
# Brillo medio (mas cercanos a la mediana)
for s in sorted(_stats, key=lambda s: abs(s["b"]-_b_med)):
    if len([c for c in CASOS if c[1]=="medium"]) >= N_MED: break
    _add("Well-exposed (mid brightness)", "medium", s)
# Mayor area de especulares
for s in sorted(_stats, key=lambda s: -s["spec"]):
    if len([c for c in CASOS if c[1]=="specular"]) >= N_SPEC: break
    _add("High specular reflections", "specular", s)

print(f"Casos seleccionados: {len(CASOS)}")
for label, slug, s in CASOS:
    print(f"  [{slug:8}] {s['ds']}/{s['kf']} frame {s['fid']}  L={s['b']:.1f}  spec={s['spec']:.1f}%")

methods = list(CORRECTIONS.keys())   # incluye endosttn si esta disponible

# endosttn necesita (img, ds, kf, fid); el resto solo img
def _apply(m, img, ds, kf, fid):
    return CORRECTIONS[m](img, ds=ds, kf=kf, fid=fid) if m == "endosttn" else CORRECTIONS[m](img)

import os
os.makedirs(REPO_ROOT/"outcomes"/"avance6", exist_ok=True)

# Visualizar DISPARIDAD (1/depth): convencion de depth. Cerca = claro, lejos = oscuro.
def _pred_disp_vis(depth):
    disp = 1.0/np.clip(depth, 1e-6, None)
    return np.clip(disp, 0, np.percentile(disp, 95))

# GT de SCARED es DISPERSO (huecos del sensor): enmascarar invalidos -> gris; escala robusta.
def _gt_depth_masked(gt):
    g = gt.astype(np.float32).copy()
    valido = np.isfinite(g) & (g > 1.0) & (g < 200.0)
    disp = np.zeros_like(g)
    disp[valido] = 1.0/g[valido]
    if valido.sum() > 0:
        disp = np.clip(disp, 0, np.percentile(disp[valido], 95))
    return ma.masked_array(disp, mask=~valido), valido.mean()*100

_cmap = plt.cm.magma.copy(); _cmap.set_bad(color="#cccccc")

# Contador por regimen para nombrar PNGs unicos: avance6_under_1.png, ...
_slug_count = {}
GENERATED = []   # rutas en orden, para la celda de HTML

for label, slug, s in CASOS:
    ds, kf, fid = s["ds"], s["kf"], s["fid"]
    img, gt = load_split_frame(ds, kf, fid)
    if img is None: continue
    ncol = 1 + len(methods)
    fig, axes = plt.subplots(2, ncol, figsize=(3.6*ncol, 7))
    fig.suptitle(f"{label}  -  MonoIIT  -  {ds}/{kf} frame {fid}  (L={s['b']:.0f}, spec={s['spec']:.0f}%)",
                 fontsize=18, fontweight="bold")

    # Columna 0 = Ground Truth
    _gt_vis, _cov = _gt_depth_masked(gt)
    axes[0,0].imshow(img); axes[0,0].set_title("original image", fontsize=16, fontweight="bold"); axes[0,0].axis("off")
    axes[1,0].imshow(_gt_vis, cmap=_cmap)
    axes[1,0].set_title(f"GROUND TRUTH\n({_cov:.0f}% with data)", fontsize=14, fontweight="bold"); axes[1,0].axis("off")
    axes[0,0].text(-0.14, 0.5, "input", rotation=90, va="center", ha="center",
                   transform=axes[0,0].transAxes, fontsize=15, fontweight="bold")
    axes[1,0].text(-0.14, 0.5, "depth\n(near = bright)", rotation=90, va="center", ha="center",
                   transform=axes[1,0].transAxes, fontsize=13, fontweight="bold")

    for j, m in enumerate(methods, start=1):
        corr = _apply(m, img, ds, kf, fid)
        depth = predict_depth(corr)
        axes[0,j].imshow(corr); axes[0,j].set_title(m, fontsize=16, fontweight="bold"); axes[0,j].axis("off")
        axes[1,j].imshow(_pred_disp_vis(depth), cmap="magma"); axes[1,j].axis("off")

    plt.tight_layout()
    _slug_count[slug] = _slug_count.get(slug, 0) + 1
    _name = f"avance6_{slug}_{_slug_count[slug]}.png"
    _out = REPO_ROOT/"outcomes"/"avance6"/_name
    fig.savefig(_out, dpi=130, bbox_inches="tight")
    GENERATED.append((label, _name, str(_out)))
    print(f"Guardado: {_out}")
    plt.show()

# Compatibilidad: mantener tambien avance6_under.png / avance6_over.png (1er caso de cada uno)
import shutil
for slug in ("under","over"):
    src = REPO_ROOT/"outcomes"/"avance6"/f"avance6_{slug}_1.png"
    if src.exists():
        shutil.copy(src, REPO_ROOT/"outcomes"/"avance6"/f"avance6_{slug}.png")


---
## Lectura de los resultados

Los mapas se muestran en **disparidad** (1/profundidad): **lo cercano es claro, lo lejano oscuro**. La columna **GROUND TRUTH** es la referencia real del sensor SCARED; cada predicción debe parecerse a ella.

**Caso subexpuesto (*under*).** En las zonas oscuras la imagen `none` aporta poca textura, por lo que el mapa de profundidad tiende a aplanarse o a colapsar. IAT **levanta la luminancia** y recupera estructura; conviene observar si la disparidad predicha se **acerca al ground truth** (recupera el gradiente de superficie) sin introducir ruido en las sombras.

**Caso sobreexpuesto (*over*).** En las regiones quemadas la información está saturada. Aquí interesa ver si el *enhancement* **atenúa el brillo** y devuelve geometría plausible (coherente con el GT) en las zonas especulares, o si —al contrario— amplifica artefactos.

**Caso bien expuesto (brillo medio).** Sirve de control: con iluminación normal el modelo ya estima bien la geometría, por lo que el realce debería **mover poco** la predicción. Si la altera, es señal de que el enhancement saca al modelo de su distribución.

**Caso con reflejos especulares.** Es el régimen donde el inpainting temporal **endosttn** tiene más sentido: al reconstruir el tejido bajo el reflejo, puede limpiar el mapa de profundidad justo donde la saturación lo rompía.

**Relación con las métricas.** Estas observaciones cualitativas complementan la tabla del estudio: un *enhancement* puede mejorar el AbsRel **global** pero deteriorar un régimen específico (o viceversa). El análisis por exposición, contra el ground truth, ayuda a explicar *por qué* IAT tiende a ayudar (corrección suave que preserva la estructura) mientras que un realce agresivo del color/contraste puede degradar la prediccion.

> **Nota.** Los fotogramas se eligen automáticamente (por brillo medio y por área de especulares); al reejecutar sobre el mismo split son reproducibles. **endosttn** (inpainting temporal de especularidades) sí se incluye en esta vista, usando la secuencia completa del *keyframe*.

In [ ]:
# --- Generar HTML de entrega embebiendo TODAS las figuras (autocontenido) ---
import os, base64
_out_dir = REPO_ROOT/"docs"/"reports"; os.makedirs(_out_dir, exist_ok=True)

def _b64(p):
    with open(p,"rb") as f: return base64.b64encode(f.read()).decode()

# Descripcion por regimen (slug -> (titulo seccion, parrafo))
_DESC = {
    "under":    ("Subexpuesto (under)",
                 "Fotogramas mas oscuros del split. En zonas en sombra la imagen <code>none</code> aporta poca textura y la profundidad tiende a aplanarse; interesa ver si el realce recupera estructura sin introducir ruido."),
    "over":     ("Sobreexpuesto (over)",
                 "Fotogramas mas brillantes del split. La saturacion borra textura; interesa ver si el realce atenua el brillo y devuelve geometria plausible, o si amplifica artefactos."),
    "medium":   ("Bien expuesto (brillo medio)",
                 "Fotogramas de brillo cercano a la mediana, como referencia del comportamiento en condiciones normales: aqui el realce deberia mover poco la prediccion."),
    "specular": ("Reflejos especulares marcados",
                 "Fotogramas con mayor area de reflejos especulares. Es el regimen donde el inpainting temporal (<code>endosttn</code>) podria ayudar mas, al reconstruir el tejido bajo el reflejo."),
}
_ORDER = ["under", "over", "medium", "specular"]

# Agrupar las figuras generadas por regimen (GENERATED viene de la celda de visualizacion)
_by_slug = {}
for _label, _name, _full in GENERATED:
    _slug = _name.replace("avance6_","").rsplit("_",1)[0]   # avance6_under_2.png -> under
    _by_slug.setdefault(_slug, []).append(_full)

_secs = ""
for _slug in _ORDER:
    if _slug not in _by_slug: continue
    _titulo, _parrafo = _DESC[_slug]
    _secs += f'<h2>{_titulo}</h2>\n<p class="desc">{_parrafo}</p>\n'
    for _full in _by_slug[_slug]:
        if os.path.exists(_full):
            _secs += '<img src="data:image/png;base64,' + _b64(_full) + '" style="width:100%;max-width:1280px;border:1px solid #ddd;border-radius:8px;margin:12px 0;">\n'

_html_doc = f"""<!doctype html><html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Avance 6 - Equipo 52</title>
<style>
 body{{font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;max-width:1320px;margin:0 auto;padding:0 20px 60px;color:#1a2230;line-height:1.6}}
 header{{background:linear-gradient(135deg,#0036a3,#012a73);color:#fff;margin:0 -20px 24px;padding:30px 28px;border-radius:0 0 14px 14px}}
 header .tag{{display:inline-block;background:rgba(255,255,255,.16);padding:4px 12px;border-radius:20px;font-size:.74em;text-transform:uppercase;letter-spacing:.04em;margin-bottom:12px}}
 header h1{{margin:0;font-size:1.6em}}
 header p{{margin:8px 0 0;color:#cdd9f2;max-width:820px}}
 h2{{color:#0036a3;border-bottom:2px solid #E0A800;padding-bottom:4px;margin-top:36px;font-size:1.3em}}
 .desc{{color:#33404f;max-width:900px}}
 .note{{background:#f7f9fc;border-left:4px solid #E0A800;padding:12px 16px;border-radius:0 8px 8px 0;margin:18px 0}}
 .legend{{display:flex;align-items:center;gap:12px;margin-top:10px;font-size:.9em;color:#5b6573}}
 .bar{{height:14px;width:220px;border-radius:7px;border:1px solid #e4e8ef;background:linear-gradient(90deg,#000004,#3b0f70,#8c2981,#de4968,#fe9f6d,#fcfdbf)}}
 code{{background:#eef1f6;padding:1px 6px;border-radius:4px;color:#1f3a8a}}
 footer{{margin-top:40px;padding-top:18px;border-top:1px solid #e4e8ef;font-size:.85em;color:#5b6573}}
</style></head><body>
<header>
 <span class="tag">Proyecto Integrador TC5035.10 &middot; Equipo 52</span>
 <h1>Avance 6 &middot; Image Enhancement bajo distintos regimenes de iluminacion</h1>
 <p>Inspeccion cualitativa del modelo de menor error (MonoIIT, AbsRel 0.0554) sobre fotogramas representativos del split: subexpuestos, sobreexpuestos, bien expuestos y con reflejos especulares.</p>
</header>

<p>Cada figura tiene <b>dos filas</b> (entrada RGB y profundidad predicha) y varias columnas: el <b>ground truth</b>
del sensor SCARED, la imagen sin tocar (<code>none</code>) y los enhancements (<code>endolmspec</code>,
<code>iat</code>, <code>endosttn</code>). Los titulos de las figuras estan en ingles.</p>

<div class="note"><b>Codigo de color de la profundidad.</b> Colormap <b>magma</b>: <b>amarillo = cerca</b>,
<b>morado/negro = lejos</b>. El ground truth se reescala de forma robusta; las zonas grises del GT son pixeles
que el sensor no midio.
<div class="legend"><span>cerca</span><span class="bar"></span><span>lejos</span></div></div>

{_secs}
<h2>Lectura general</h2>
<p>Estas vistas cualitativas complementan la tabla del estudio: un enhancement puede mejorar el AbsRel global
pero comportarse distinto en un regimen especifico. <b>IAT</b> tiende a realzar sin alterar la geometria;
<b>endosttn</b> destaca solo donde hay reflejos especulares; en penumbra el modelo ya es robusto de base.</p>

<footer>Proyecto Integrador TC5035.10 &middot; Equipo 52. MonoIIT con pesos SCARED (Dr. Ricardo Espinosa Loera).
Fotogramas seleccionados automaticamente por brillo medio (canal L de CIELAB) y por area de reflejos especulares,
sobre el split oficial de AF-SfMLearner.</footer>
</body></html>"""

_html_path = _out_dir/"avance6_visualizacion.html"
with open(_html_path,"w",encoding="utf-8") as f: f.write(_html_doc)
print(f"HTML de entrega generado: {_html_path}")
print(f"  figuras embebidas: {sum(len(v) for v in _by_slug.values())}")
print("Subelo al repo: git add docs/reports/avance6_visualizacion.html outcomes/avance6/")


In [10]:
import subprocess
from getpass import getpass
token = getpass("GitHub token: ")
subprocess.run(["git","-C",str(REPO_ROOT),"remote","set-url","origin",
    f"https://{token}@github.com/jmtoral/proyecto_integrador_52.git"])
subprocess.run(["git","-C",str(REPO_ROOT),"add",
    "docs/reports/avance6_visualizacion.html","outcomes/avance6"])
subprocess.run(["git","-C",str(REPO_ROOT),"commit","-m","Add: HTML y PNG del Avance6"])
subprocess.run(["git","-C",str(REPO_ROOT),"pull","--rebase","origin","main"])
subprocess.run(["git","-C",str(REPO_ROOT),"push","origin","main"])

CompletedProcess(args=['git', '-C', '/content/repo_52', 'push', 'origin', 'main'], returncode=128)